# Notebook 04 · Part 3 — Scalable Processing: Stage 1 Pilot
**FDIC bank–quarter panel · Polars Lazy + pandas + PySpark local**

**Phase 1 Objective:** Replicate the key transformations from Notebook 03B for the **2019–2020** period, prior to conducting a full benchmark. These tests must rely exclusively on data already present in Drive; they must **not** call the FDIC API, nor should they modify `data_raw`, `data_storage`, `clean_panel_v1`, or the outcome history files from 03C.

## 0. Install dependencies & environment
Install the required libraries, connect to the project storage, and verify that the input datasets are available.

In [ ]:
%pip -q install "polars>=1.32,<2" "pyarrow>=17,<25" "pandas>=2.2,<3" "pyspark>=3.5,<4.1" "psutil>=5.9"

In [ ]:
import os, sys, json, time, shutil, platform, zipfile, tempfile, contextlib, io
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import psutil

root_override = os.environ.get('FDIC_PROJECT_ROOT')
if root_override:
    PROJECT_ROOT = Path(root_override).expanduser().resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')

FIN_DIR = PROJECT_ROOT / 'data_storage/financials/year_partitioned_full'
CLEAN_DIR = PROJECT_ROOT / 'data_processed/clean_panel_v1'
REPORT_DIR = PROJECT_ROOT / 'outputs/part3/stage1_pilot_v1'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# These are the actual Parquet paths used by Part 2. No writes to these inputs.
assert FIN_DIR.is_dir(), f'Part 1 Parquet not found: {FIN_DIR}'
assert CLEAN_DIR.is_dir(), f'Part 2 clean panel not found: {CLEAN_DIR}'
assert REPORT_DIR.resolve() != FIN_DIR.resolve()
assert REPORT_DIR.resolve() != CLEAN_DIR.resolve()
print('Project:', PROJECT_ROOT)
print('Polars:', pl.__version__, '| pandas:', pd.__version__)
print('CPU:', psutil.cpu_count(logical=True), '| RAM GiB:', round(psutil.virtual_memory().total/1024**3, 2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/FDIC_Bank_Failure_Project
Polars: 1.35.2 | pandas: 2.2.3
CPU: 2 | RAM GiB: 12.67


## 1. Pilot scope, reproducible input & safety gates
Select the 2019–2020 data, validate its schema and size, and copy the same input files to local storage for all three processing engines.

In [ ]:
PILOT_YEARS = (2019, 2020)
RUN_SPARK = True  # If Java/Spark setup fails, set False to complete Polars+pandas pilot first.
MAX_PILOT_ROWS = 150_000  # Safety guard; do not silently run on all 1.68 million rows.

BALANCE_NONNEGATIVE_FIELDS = [
    'ASSET','DEP','DEPDOM','COREDEP','DEPINS','DEPUNINS','CHBAL',
    'SC','LNLSGR','LNLSNET','P3ASSET','P9ASSET','NAASSET'
]
RATIO_REVIEW_ABS_THRESHOLDS = {
    'ROA':100.0, 'ROE':1000.0, 'NIMY':100.0, 'RBC1AAJ':1000.0
}
DENOMINATOR_REVIEW_FIELDS = ['ASSET', 'DEP', 'LNLSGR']
KEYS = ['CERT', 'REPDTE']

source_files = [FIN_DIR / f'YEAR={year}' / 'data.parquet' for year in PILOT_YEARS]
reference_files = [CLEAN_DIR / f'YEAR={year}' / 'data.parquet' for year in PILOT_YEARS]
for p in source_files + reference_files:
    assert p.is_file(), f'Missing Part 1/2 file: {p}'

first_schema = pq.ParquetFile(source_files[0]).schema_arrow
original_columns = first_schema.names
assert len(original_columns) == 47, f'Expected 47 original fields, found {len(original_columns)}'
required = set(KEYS + BALANCE_NONNEGATIVE_FIELDS + list(RATIO_REVIEW_ABS_THRESHOLDS))
assert required.issubset(set(original_columns)), f'Missing required fields: {sorted(required-set(original_columns))}'
for p in source_files:
    assert pq.ParquetFile(p).schema_arrow.equals(first_schema), f'Schema mismatch: {p}'

source_inventory = pd.DataFrame([
    {'year':y, 'file':str(p.relative_to(PROJECT_ROOT)),
     'rows':int(pq.ParquetFile(p).metadata.num_rows),
     'bytes':p.stat().st_size}
    for y,p in zip(PILOT_YEARS,source_files)
])
assert 0 < int(source_inventory.rows.sum()) <= MAX_PILOT_ROWS, 'Pilot size outside configured safety range'
assert source_inventory.year.is_unique

# The local stage is expendable and never references a source destination.
LOCAL_STAGE = Path('/content/fdic_part3_stage1') if Path('/content').is_dir() else Path(tempfile.gettempdir()) / 'fdic_part3_stage1'
LOCAL_STAGE.mkdir(parents=True,exist_ok=True)
local_files = []
for year,p in zip(PILOT_YEARS,source_files):
    dst = LOCAL_STAGE / f'raw_{year}.parquet'
    # Always refresh the local stage; equal file sizes do not guarantee identical contents.
    shutil.copy2(p,dst)
    assert dst.stat().st_size == p.stat().st_size
    assert pq.ParquetFile(dst).metadata.num_rows == pq.ParquetFile(p).metadata.num_rows
    local_files.append(dst)
print(source_inventory.to_string(index=False))
print(f'Staged {sum(source_inventory.rows):,} rows on Colab local disk:', LOCAL_STAGE)

 year                                                                 file  rows   bytes
 2019 data_storage/financials/year_partitioned_full/YEAR=2019/data.parquet 21366 4559316
 2020 data_storage/financials/year_partitioned_full/YEAR=2020/data.parquet 20478 4348850
Staged 41,844 rows on Colab local disk: /content/fdic_part3_stage1


## 2. Canonical business logic (identical to 03B)
**T1 — Filter/projection:** Select only the necessary columns and years. **T2 — Cleaning/QA:** Retain the original data, generate 13 `*_QA_VALID` flags, and compute 3 QA counters in accordance with the 03B policy. **T3 — Join:** Perform a join with the quarterly calendar (excluding future-dated information). **T4 — Window:** Calculate the number of missing quarters between two *actual consecutive* reports for the same CERT. **T5 — GroupBy:** Aggregate data by quarter.

In the pilot phase, T3 serves as a calendar join intended for testing the execution plan; it is **not** a join with outcome data (Failures/History). The distinction between the `lag` of the preceding record and the preceding quarter is handled via `Q_ORD`.

In [ ]:
# Use a dimension table built from known quarter-end dates, not from future outcomes.
quarter_dim = pd.DataFrame([
    {'REPDTE':per.end_time.strftime('%Y%m%d'),
     'CAL_YEAR':int(per.year), 'CAL_QUARTER':int(per.quarter)}
    for per in pd.period_range(f'{min(PILOT_YEARS)}Q1', f'{max(PILOT_YEARS)}Q4', freq='Q')
])
assert quarter_dim.REPDTE.is_unique

RAW_KEEP = list(dict.fromkeys(
    KEYS + BALANCE_NONNEGATIVE_FIELDS + list(RATIO_REVIEW_ABS_THRESHOLDS)
    + ['EQ','NETINC']
))
# These flags must match the source Part 2 cleaning pipeline exactly.
QA_FIELDS = ['QA_NEGATIVE_BALANCE_COUNT','QA_RATIO_REVIEW_COUNT','QA_ZERO_DENOM_COUNT']
VALID_FIELDS = [f'{c}_QA_VALID' for c in BALANCE_NONNEGATIVE_FIELDS]
FINAL_FIELDS = RAW_KEEP + VALID_FIELDS + QA_FIELDS + [
    'CAL_YEAR','CAL_QUARTER','Q_ORD','PREV_Q_ORD','GAP_QUARTERS'
]
assert len(FINAL_FIELDS) == len(set(FINAL_FIELDS)), 'Duplicate projected column'

print('Input projection columns:', len(RAW_KEEP), '| QA_VALID fields:', len(VALID_FIELDS))
print('Calendar dimension rows:', len(quarter_dim))

Input projection columns: 21 | QA_VALID fields: 13
Calendar dimension rows: 8


## 3. Polars Lazy execution + optimized plans
Implement T1–T5 using Polars lazy queries, inspect optimized execution plans, and generate the processed panel and quarterly summaries.

In [ ]:
def polars_core(raw_lf):
    df = raw_lf.select(RAW_KEEP)
    flags = [
        (pl.col(c) < 0).fill_null(False).cast(pl.Int32)
        for c in BALANCE_NONNEGATIVE_FIELDS
    ]
    ratio = [
        (pl.col(c).abs() > bound).fill_null(False).cast(pl.Int32)
        for c,bound in RATIO_REVIEW_ABS_THRESHOLDS.items()
    ]
    denom = [
        (pl.col(c) == 0).fill_null(False).cast(pl.Int32)
        for c in DENOMINATOR_REVIEW_FIELDS
    ]
    df = df.with_columns([
        *[pl.when(pl.col(c) < 0).then(None).otherwise(pl.col(c)).alias(f'{c}_QA_VALID')
          for c in BALANCE_NONNEGATIVE_FIELDS],
        pl.sum_horizontal(flags).cast(pl.Int32).alias('QA_NEGATIVE_BALANCE_COUNT'),
        pl.sum_horizontal(ratio).cast(pl.Int32).alias('QA_RATIO_REVIEW_COUNT'),
        pl.sum_horizontal(denom).cast(pl.Int32).alias('QA_ZERO_DENOM_COUNT'),
    ])
    return df

def polars_final(core_lf):
    cal_lf = pl.from_pandas(quarter_dim).lazy()
    df = core_lf.join(cal_lf, on='REPDTE', how='left', validate='m:1')
    df = df.with_columns(
        (pl.col('CAL_YEAR')*4 + pl.col('CAL_QUARTER')-1).cast(pl.Int32).alias('Q_ORD')
    )
    df = df.sort(['CERT','REPDTE']).with_columns(
        pl.col('Q_ORD').shift(1).over('CERT').alias('PREV_Q_ORD')
    )
    df = df.with_columns(
        pl.when(pl.col('PREV_Q_ORD').is_null()).then(None)
        .otherwise((pl.col('Q_ORD')-pl.col('PREV_Q_ORD')-1).clip(lower_bound=0))
        .cast(pl.Int32).alias('GAP_QUARTERS')
    )
    return df.select(FINAL_FIELDS)

def polars_quarter_agg(final_lf):
    return final_lf.group_by('REPDTE').agg([
        pl.len().alias('BANK_COUNT'),
        pl.col('ASSET').sum().alias('ASSET_TOTAL'),
        pl.col('QA_NEGATIVE_BALANCE_COUNT').sum().alias('NEG_BALANCE_FLAGS'),
        (pl.col('GAP_QUARTERS')>0).fill_null(False).cast(pl.Int64).sum().alias('REPORTING_GAP_BANKS'),
    ]).sort('REPDTE')

raw_lazy = pl.scan_parquet([str(p) for p in local_files], hive_partitioning=False)
pl_core_lf = polars_core(raw_lazy)
pl_final_lf = polars_final(pl_core_lf)
pl_agg_lf = polars_quarter_agg(pl_final_lf)

# Separate demonstration of explicit predicate and projection pushdown.
t1_filter = f'{max(PILOT_YEARS)}0101'
t1_end = f'{max(PILOT_YEARS)}1231'
t1_lf = raw_lazy.filter(pl.col('REPDTE').is_between(pl.lit(t1_filter), pl.lit(t1_end))).select(['CERT','REPDTE','ASSET'])
polars_t1_plan = t1_lf.explain(optimized=True)
polars_full_plan = pl_agg_lf.explain(optimized=True)
(REPORT_DIR/'stage1_polars_T1_optimized_plan.txt').write_text(polars_t1_plan,encoding='utf-8')
(REPORT_DIR/'stage1_polars_T2_T5_optimized_plan.txt').write_text(polars_full_plan,encoding='utf-8')

print('T1 plan (first 3000 chars):\n',polars_t1_plan[:3000])
print('\nT2–T5 optimized plan (first 3000 chars):\n',polars_full_plan[:3000])
pl_final = pl_final_lf.collect()
pl_agg = pl_agg_lf.collect()
pl_t1 = t1_lf.collect()
assert pl_final.height == int(source_inventory.rows.sum())
assert pl_final.select(pl.struct(KEYS).n_unique()).item() == pl_final.height
assert pl_final['CAL_YEAR'].null_count() == 0
print('Polars successful:',pl_final.height,'rows | quarterly aggregates:',pl_agg.height)

T1 plan (first 3000 chars):
 Parquet SCAN [/content/fdic_part3_stage1/raw_2019.parquet, /content/fdic_part3_stage1/raw_2020.parquet]
PROJECT 3/47 COLUMNS
SELECTION: col("REPDTE").is_between(["20200101", "20201231"])
ESTIMATED ROWS: 42732

T2–T5 optimized plan (first 3000 chars):
 SORT BY [col("REPDTE")]
  AGGREGATE[maintain_order: false]
    [len().alias("BANK_COUNT"), col("ASSET").sum().alias("ASSET_TOTAL"), col("QA_NEGATIVE_BALANCE_COUNT").sum().alias("NEG_BALANCE_FLAGS"), [(col("GAP_QUARTERS")) > (0)].fill_null([false]).strict_cast(Int64).sum().alias("REPORTING_GAP_BANKS")] BY [col("REPDTE")]
    FROM
    simple π 4/4 ["REPDTE", "ASSET", ... 2 other columns]
       WITH_COLUMNS:
       [when(col("PREV_Q_ORD").is_null()).then(null.cast(Int32)).otherwise([([(col("Q_ORD")) - (col("PREV_Q_ORD"))]) - (1)].clip_min([dyn int: 0])).alias("GAP_QUARTERS")] 
         WITH_COLUMNS:
         [col("Q_ORD").shift([dyn int: 1]).over([col("CERT")]).alias("PREV_Q_ORD")] 
          SORT BY [col("CERT"

## 4. pandas baseline: equivalent operations on exactly the same pilot files
Apply the same transformations to the identical pilot files using pandas, creating a baseline for checking the Polars results.

In [ ]:
def pandas_core(raw):
    df = raw[RAW_KEEP].copy()
    negative_counts = np.zeros(len(df),dtype=np.int32)
    ratio_counts = np.zeros(len(df),dtype=np.int32)
    denom_counts = np.zeros(len(df),dtype=np.int32)
    for c in BALANCE_NONNEGATIVE_FIELDS:
        invalid = df[c].lt(0).fillna(False)
        df[f'{c}_QA_VALID'] = df[c].mask(invalid)
        negative_counts += invalid.to_numpy(dtype=np.int32)
    for c,bound in RATIO_REVIEW_ABS_THRESHOLDS.items():
        ratio_counts += df[c].abs().gt(bound).fillna(False).to_numpy(dtype=np.int32)
    for c in DENOMINATOR_REVIEW_FIELDS:
        denom_counts += df[c].eq(0).fillna(False).to_numpy(dtype=np.int32)
    df['QA_NEGATIVE_BALANCE_COUNT'] = negative_counts
    df['QA_RATIO_REVIEW_COUNT'] = ratio_counts
    df['QA_ZERO_DENOM_COUNT'] = denom_counts
    return df

def pandas_final(core):
    df = core.merge(quarter_dim, on='REPDTE', how='left',validate='many_to_one')
    assert df.CAL_YEAR.notna().all(), 'Date not matched to calendar'
    df['Q_ORD'] = df['CAL_YEAR']*4 + df['CAL_QUARTER'] - 1
    df = df.sort_values(['CERT','REPDTE'],kind='mergesort').reset_index(drop=True)
    df['PREV_Q_ORD'] = df.groupby('CERT',sort=False)['Q_ORD'].shift(1)
    df['GAP_QUARTERS'] = (df.Q_ORD - df.PREV_Q_ORD - 1).clip(lower=0)
    return df[FINAL_FIELDS]

def pandas_quarter_agg(df):
    a = df.assign(_HAS_GAP=(df.GAP_QUARTERS>0).astype('int64'))
    return a.groupby('REPDTE',as_index=False).agg(
        BANK_COUNT=('CERT','size'),
        ASSET_TOTAL=('ASSET','sum'),
        NEG_BALANCE_FLAGS=('QA_NEGATIVE_BALANCE_COUNT','sum'),
        REPORTING_GAP_BANKS=('_HAS_GAP','sum')
    ).sort_values('REPDTE').reset_index(drop=True)

raw_pd = pd.concat([
    pd.read_parquet(p,columns=RAW_KEEP,engine='pyarrow')
    for p in local_files
],ignore_index=True)
pd_final = pandas_final(pandas_core(raw_pd))
pd_agg = pandas_quarter_agg(pd_final)
pd_t1 = raw_pd.loc[
    raw_pd.REPDTE.between(t1_filter,t1_end), ['CERT','REPDTE','ASSET']
].copy()
assert len(pd_final) == len(pl_final) == int(source_inventory.rows.sum())
print('pandas successful:',len(pd_final),'rows | aggregates:',len(pd_agg))

pandas successful: 41844 rows | aggregates: 8


## 5. Automated parity: Polars vs pandas **and** existing 03B clean panel
Compare Polars and pandas outputs by bank-quarter keys, values, and missingness. Also verify that the quality-control fields match the cleaned panel from Notebook 03B.

In [ ]:
PARITY_RECORDS = []

def check_frame_parity(left, right, keys, compare_cols, label):
    a = left.sort_values(keys).reset_index(drop=True).copy()
    b = right.sort_values(keys).reset_index(drop=True).copy()
    assert set(keys + compare_cols).issubset(a.columns), 'Missing columns on left'
    assert set(keys + compare_cols).issubset(b.columns), 'Missing columns on right'
    assert not a.duplicated(keys).any(), f'Left duplicate keys: {label}'
    assert not b.duplicated(keys).any(), f'Right duplicate keys: {label}'
    key_match = (len(a)==len(b) and a[keys].equals(b[keys]))
    PARITY_RECORDS.append({'comparison':label,'field':'__KEY_AND_ROWS__',
                           'passed':bool(key_match),'mismatched_rows':0 if key_match else max(len(a),len(b))})
    if not key_match:
        raise AssertionError(f'{label}: key or row mismatch; fix before numeric comparisons')
    for c in compare_cols:
        av = pd.to_numeric(a[c],errors='coerce').to_numpy(dtype='float64',na_value=np.nan)
        bv = pd.to_numeric(b[c],errors='coerce').to_numpy(dtype='float64',na_value=np.nan)
        exact = c in QA_FIELDS or c in {'CAL_YEAR','CAL_QUARTER','Q_ORD',
                                        'PREV_Q_ORD','GAP_QUARTERS','BANK_COUNT',
                                        'NEG_BALANCE_FLAGS','REPORTING_GAP_BANKS'}
        eq = np.isclose(av,bv,atol=0 if exact else 1e-6,
                        rtol=0 if exact else 1e-8,equal_nan=True)
        misses = int((~eq).sum())
        PARITY_RECORDS.append({'comparison':label,'field':c,
                               'passed':misses==0,'mismatched_rows':misses})
    return all(r['passed'] for r in PARITY_RECORDS if r['comparison']==label)

pl_final_pd = pl_final.to_pandas()
pl_agg_pd = pl_agg.to_pandas()
pl_t1_pd = pl_t1.to_pandas()

assert check_frame_parity(pl_t1_pd,pd_t1,KEYS,['ASSET'],'Polars vs pandas | T1 projection/filter')
assert check_frame_parity(pl_final_pd,pd_final,KEYS,
                           VALID_FIELDS+QA_FIELDS+['CAL_YEAR','CAL_QUARTER','Q_ORD',
                                                 'PREV_Q_ORD','GAP_QUARTERS'],
                           'Polars vs pandas | T2–T4 cleaning/join/window')
assert check_frame_parity(pl_agg_pd,pd_agg,['REPDTE'],
                           ['BANK_COUNT','ASSET_TOTAL','NEG_BALANCE_FLAGS','REPORTING_GAP_BANKS'],
                           'Polars vs pandas | T5 groupby')

# Reference verification: the 03B accepted QA fields must match in every sampled bank-quarter.
reference_keep = KEYS + VALID_FIELDS + QA_FIELDS
reference_pd = pd.concat([
    pd.read_parquet(p,columns=reference_keep,engine='pyarrow')
    for p in reference_files
],ignore_index=True)
assert check_frame_parity(pl_final_pd,reference_pd,KEYS,
                          VALID_FIELDS+QA_FIELDS,'Polars vs 03B clean panel | QA parity')
print('All Polars–pandas / 03B parity checks PASSED:',len(PARITY_RECORDS))

All Polars–pandas / 03B parity checks PASSED: 46


## 6. PySpark local: reproduce T1–T5 and interpret physical plan
Reproduce T1–T5 in local PySpark, inspect its physical execution plan, and compare the results with Polars. This tests correctness on one machine, not distributed-cluster performance.

In [ ]:
SPARK_RAN = False
SPARK_PARITY_OK = False
spark_plan = None
if RUN_SPARK:
    from pyspark.sql import SparkSession, Window
    from pyspark.sql import functions as F

    spark = (SparkSession.builder.master('local[2]')
             .appName('FDIC_Part3_Stage1_Pilot')
             .config('spark.sql.shuffle.partitions','8')
             .config('spark.sql.adaptive.enabled','true')
             .getOrCreate())
    spark.sparkContext.setLogLevel('WARN')

    spark_raw = spark.read.parquet(*[str(p) for p in local_files]).select(*RAW_KEEP)
    # Guard before any .toPandas() conversion.
    spark_input_count = spark_raw.count()
    assert spark_input_count == int(source_inventory.rows.sum()) <= MAX_PILOT_ROWS

    spark_core = spark_raw
    neg_exprs, ratio_exprs, zero_exprs = [], [], []
    for c in BALANCE_NONNEGATIVE_FIELDS:
        invalid = (F.col(c)<0)
        spark_core = spark_core.withColumn(f'{c}_QA_VALID',
                         F.when(invalid,F.lit(None)).otherwise(F.col(c)))
        neg_exprs.append(F.when(invalid,F.lit(1)).otherwise(F.lit(0)))
    for c,lim in RATIO_REVIEW_ABS_THRESHOLDS.items():
        ratio_exprs.append(F.when(F.abs(F.col(c))>lim,F.lit(1)).otherwise(F.lit(0)))
    for c in DENOMINATOR_REVIEW_FIELDS:
        zero_exprs.append(F.when(F.col(c)==0,F.lit(1)).otherwise(F.lit(0)))

    def spark_sum_expr(parts):
        result = F.lit(0)
        for e in parts:
            result = result+e
        return result

    spark_core = (spark_core
       .withColumn('QA_NEGATIVE_BALANCE_COUNT',spark_sum_expr(neg_exprs))
       .withColumn('QA_RATIO_REVIEW_COUNT',spark_sum_expr(ratio_exprs))
       .withColumn('QA_ZERO_DENOM_COUNT',spark_sum_expr(zero_exprs)))
    spark_cal = spark.createDataFrame(
        [(str(r.REPDTE),int(r.CAL_YEAR),int(r.CAL_QUARTER))
         for r in quarter_dim.itertuples(index=False)],
        schema='REPDTE string, CAL_YEAR int, CAL_QUARTER int'
    )
    spark_joined = (spark_core
        .join(F.broadcast(spark_cal),on='REPDTE',how='left')
        .withColumn('Q_ORD',(F.col('CAL_YEAR')*4+F.col('CAL_QUARTER')-1).cast('int')))
    w = Window.partitionBy('CERT').orderBy('REPDTE')
    spark_final = (spark_joined
        .withColumn('PREV_Q_ORD',F.lag('Q_ORD',1).over(w))
        .withColumn('GAP_QUARTERS',
            F.when(F.col('PREV_Q_ORD').isNull(),F.lit(None).cast('int'))
             .otherwise(F.greatest(F.col('Q_ORD')-F.col('PREV_Q_ORD')-1,F.lit(0))))
        .select(*FINAL_FIELDS))
    spark_agg = spark_final.groupBy('REPDTE').agg(
        F.count('*').alias('BANK_COUNT'),
        F.sum('ASSET').alias('ASSET_TOTAL'),
        F.sum('QA_NEGATIVE_BALANCE_COUNT').alias('NEG_BALANCE_FLAGS'),
        F.sum(F.when(F.col('GAP_QUARTERS')>0,1).otherwise(0)).alias('REPORTING_GAP_BANKS'),
    )
    spark_t1 = (spark_raw
        .filter(F.col('REPDTE').between(t1_filter,t1_end))
        .select('CERT','REPDTE','ASSET'))

    # Physical plan is captured from Spark's supported formatted explain API.
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        spark_agg.explain(mode='formatted')
    spark_plan = buf.getvalue()
    (REPORT_DIR/'stage1_spark_T2_T5_physical_plan.txt').write_text(spark_plan,encoding='utf-8')
    print('Spark formatted physical plan (first 4500 chars):\n',spark_plan[:4500])

    spark_final_pd = spark_final.toPandas()
    spark_agg_pd = spark_agg.toPandas()
    spark_t1_pd = spark_t1.toPandas()
    assert check_frame_parity(pl_t1_pd,spark_t1_pd,KEYS,['ASSET'],
                              'Polars vs Spark | T1 projection/filter')
    assert check_frame_parity(pl_final_pd,spark_final_pd,KEYS,
                              VALID_FIELDS+QA_FIELDS+['CAL_YEAR','CAL_QUARTER','Q_ORD',
                                                 'PREV_Q_ORD','GAP_QUARTERS'],
                              'Polars vs Spark | T2–T4 cleaning/join/window')
    assert check_frame_parity(pl_agg_pd,spark_agg_pd,['REPDTE'],
                              ['BANK_COUNT','ASSET_TOTAL','NEG_BALANCE_FLAGS','REPORTING_GAP_BANKS'],
                              'Polars vs Spark | T5 groupby')
    SPARK_RAN = True
    SPARK_PARITY_OK = True
    print('Spark parity PASSED | local master:',spark.sparkContext.master)
else:
    print('PySpark skipped deliberately. Pilot incomplete until Spark parity is run.')

Spark formatted physical plan (first 4500 chars):
 == Physical Plan ==
AdaptiveSparkPlan (15)
+- HashAggregate (14)
   +- Exchange (13)
      +- HashAggregate (12)
         +- Project (11)
            +- Window (10)
               +- Sort (9)
                  +- Exchange (8)
                     +- Project (7)
                        +- BroadcastHashJoin LeftOuter BuildRight (6)
                           :- Project (2)
                           :  +- Scan parquet  (1)
                           +- BroadcastExchange (5)
                              +- Filter (4)
                                 +- Scan ExistingRDD (3)


(1) Scan parquet 
Output [15]: [CERT#0L, REPDTE#1, ASSET#3, DEP#4, DEPDOM#5, COREDEP#6, DEPINS#7, DEPUNINS#8, CHBAL#12, SC#13, LNLSGR#18, LNLSNET#19, P3ASSET#29, P9ASSET#30, NAASSET#31]
Batched: true
Location: InMemoryFileIndex [file:/content/fdic_part3_stage1/raw_2019.parquet, ... 1 entries]
ReadSchema: struct<CERT:bigint,REPDTE:string,ASSET:double,DEP:double,DEPDOM

## 7. Acceptance checks, reports, and Stage 2 handoff
Summarize input checks and cross-engine consistency as PASS, FAIL, or NOT_RUN. Save the validation results, execution plans, quarterly aggregates, and summary report for review before Stage 2.

In [ ]:
parity_df = pd.DataFrame(PARITY_RECORDS)
expected_checks = {
    'INPUT_2019_2020_SOURCE_FILES_VERIFIED':len(local_files)==len(PILOT_YEARS),
    'RAW_INPUT_47_FIELDS':len(original_columns)==47,
    'ROW_COUNTS_PRESERVED':len(pd_final)==len(pl_final)==int(source_inventory.rows.sum()),
    'BANK_QUARTER_KEYS_UNIQUE':not pd_final.duplicated(KEYS).any(),
    'CLEAN_REFERENCE_03B_QA_PARITY':bool(parity_df.loc[
        parity_df.comparison=='Polars vs 03B clean panel | QA parity','passed'].all()),
    'POLARS_PANDAS_PARITY':bool(parity_df.loc[
        parity_df.comparison.str.contains('Polars vs pandas'), 'passed'].all()),
    'PYSPARK_LOCAL_PARITY':SPARK_PARITY_OK if SPARK_RAN else None,
}
status_df = pd.DataFrame([
    {'gate':name,'status':'PASS' if ok is True else ('NOT_RUN' if ok is None else 'FAIL')}
    for name,ok in expected_checks.items()
])

# Confirm pilot totals and gap counts are grounded in the actual output.
gap_count = int((pd_final.GAP_QUARTERS>0).sum())
assert int(pd_agg.REPORTING_GAP_BANKS.sum()) == gap_count
source_inventory.to_csv(REPORT_DIR/'stage1_input_inventory.csv',index=False)
parity_df.to_csv(REPORT_DIR/'stage1_parity_checks.csv',index=False)
status_df.to_csv(REPORT_DIR/'stage1_acceptance_gates.csv',index=False)
pd_agg.to_csv(REPORT_DIR/'stage1_quarter_aggregates.csv',index=False)

software = {
    'python':sys.version.split()[0], 'polars':pl.__version__,
    'pandas':pd.__version__, 'pyarrow':__import__('pyarrow').__version__,
    'pyspark':__import__('pyspark').__version__ if SPARK_RAN else None,
    'cpu_logical':psutil.cpu_count(logical=True),
    'memory_gib':round(psutil.virtual_memory().total/1024**3,3),
    'platform':platform.platform(),
    'pilot_years':list(PILOT_YEARS),
    'pilot_rows':int(source_inventory.rows.sum()),
    'execution_mode':'Colab/local single machine, not a multi-node cluster',
    'input_storage':'Copied Parquet from Drive to Colab local filesystem',
}
summary = {
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'stage':'Part3_Stage1_Correctness_Pilot',
    'pilot_rows':int(len(pd_final)),
    'pilot_years':list(PILOT_YEARS),
    'pilot_unique_banks':int(pd_final.CERT.nunique()),
    'pilot_reporting_gap_rows':gap_count,
    'window_scope_note':'Reporting gaps are calculated within the 2019–2020 pilot, not across all earlier bank history.',
    'qa_negative_balance_flag_instances':int(pd_final.QA_NEGATIVE_BALANCE_COUNT.sum()),
    'parity_fields_checked':int(len(parity_df)),
    'parity_failed_fields':int((~parity_df.passed).sum()),
    'spark_ran':bool(SPARK_RAN),
    'spark_parity_passed':bool(SPARK_PARITY_OK),
    'all_stage1_acceptance_pass':bool((status_df.status=='PASS').all()),
    'scope_limitations':'Pilot correctness only. No full benchmark, peak-memory comparison, cost or cluster claim.',
    'part2_reference_path':str(CLEAN_DIR.relative_to(PROJECT_ROOT)),
    'no_part1_part2_data_modified':True,
    'environment':software,
}
(REPORT_DIR/'stage1_summary.json').write_text(
    json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
with zipfile.ZipFile(REPORT_DIR/'FDIC_Part3_Stage1_Pilot_Reports.zip','w',zipfile.ZIP_DEFLATED) as zf:
    current_run_files = [
        'stage1_polars_T1_optimized_plan.txt',
        'stage1_polars_T2_T5_optimized_plan.txt',
        'stage1_input_inventory.csv', 'stage1_parity_checks.csv',
        'stage1_acceptance_gates.csv', 'stage1_quarter_aggregates.csv',
        'stage1_summary.json'
    ]
    if SPARK_RAN:
        current_run_files.append('stage1_spark_T2_T5_physical_plan.txt')
    for name in current_run_files:
        p = REPORT_DIR / name
        assert p.is_file(), f'Missing output: {p}'
        zf.write(p,arcname=p.name)
print('\nSTAGE 1 ACCEPTANCE:')
print(status_df.to_string(index=False))
print('\nREPORT ZIP:',REPORT_DIR/'FDIC_Part3_Stage1_Pilot_Reports.zip')
print('Stage 1 complete:',summary['all_stage1_acceptance_pass'])


STAGE 1 ACCEPTANCE:
                                 gate status
INPUT_2019_2020_SOURCE_FILES_VERIFIED   PASS
                  RAW_INPUT_47_FIELDS   PASS
                 ROW_COUNTS_PRESERVED   PASS
             BANK_QUARTER_KEYS_UNIQUE   PASS
        CLEAN_REFERENCE_03B_QA_PARITY   PASS
                 POLARS_PANDAS_PARITY   PASS
                 PYSPARK_LOCAL_PARITY   PASS

REPORT ZIP: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part3/stage1_pilot_v1/FDIC_Part3_Stage1_Pilot_Reports.zip
Stage 1 complete: True



## 8. Pilot Evaluation and Transition to Stage 2

- **Execution plans:** Examine Polars query optimizations and Spark physical execution plans.
- **Pilot scope:** Reporting gaps are calculated within the 2019–2020 observation window.
- **Correctness validation:** Verify T1–T5 output consistency across Polars, pandas, and PySpark, including QA parity with Notebook 03B.
- **Limitations:** Stage 1 evaluates processing correctness on a single machine, not full-scale performance or distributed cluster requirements.

Stage 2 extends the analysis to performance benchmarking, peak memory usage, and scalability at 25%, 50%, 75%, and 100% of the full dataset, subject to all Stage 1 acceptance checks passing.
